In [8]:
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import seaborn as sns

In [9]:
data = pd.read_csv('./data/self_supply/ele_self_supply.csv',index_col=['region'])

In [10]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25

fig,ax = plt.subplots(figsize=(10,20),dpi=400)

for k in ['FreeTrade',
          'DemandResponse',
          'HigherElecDemand',
          'HigherHeatElectrification',
          'PreRetiredThermalPower',
          'HigherDLS',
          'WithoutDLS',
          'WithoutEmisCap',
          'WithoutHPExpansion',
          'SlowerNuclearExpansion',
          'LowerVRESupply',
          'SlowerTechAdvancement',
          'DemandingDLS']:
    
    if k == 'HigherDLS' or k == 'DemandingDLS':
        sns.ecdfplot(data=data,y=k, weights='HighDLS',ax=ax,label=k)
    elif k == 'HigherElecDemand':
        sns.ecdfplot(data=data,y=k, weights='HigherElectrification',ax=ax,label=k)
    elif k == 'WithoutDLS':
        sns.ecdfplot(data=data,y=k, weights='Historical',ax=ax,label=k)
    else:
        sns.ecdfplot(data=data,y=k, weights='Base_dem',ax=ax,label=k)

sns.ecdfplot(data=data,
             y='LimitedTxExpansion', 
             weights='Base_dem',
             ax=ax,
             label='LimitedTxExpansion',
             color='red',
             lw=2.5) 
 
sns.ecdfplot(data=data,
             y='Base', 
             weights='Base_dem',
             ax=ax,
             label='Base',
             color='black',
             lw=2.5) 
        
ax.legend(loc='center right',edgecolor='black',title='Scenario')

ax.minorticks_on()

ax.tick_params(axis='y', 
               which='major', 
               length=10, 
               color='black')

ax.tick_params(axis='x', 
               which='major', 
               length=10, 
               color='black')

ax.tick_params(axis='y', 
               which='minor', 
               length=6, 
               color='black')

ax.tick_params(axis='x', 
               which='minor', 
               length=6, 
               color='black')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_ylabel('Degree of electricity self-sufficiency (0~1)',font={'size':30})

ax.set_xlim(0,1)
ax.set_ylim(0,1)

plt.savefig('./fig/ele_in_rate_ecdf.jpg',bbox_inches='tight')
plt.savefig('./fig/ele_in_rate_ecdf.pdf',bbox_inches='tight')

plt.close()

In [11]:
world = gpd.read_file('./data/shp/territory_vis.shp')
grid = gpd.read_file('./data/shp/grid_vis.shp')
regn = grid.dissolve(by='region')

In [12]:
regn = regn.reset_index()

regn_attr = pd.merge(left=regn,right=data,on=['region'])

In [13]:
def add_colorbar(fig,ax,vmax,label,vmin=0,cmap='autumn_r',loc=[0.175,0.39,0.025,0.2]):
        
    cax = ax.inset_axes(loc,transform=ax.transAxes)
    
    im = plt.cm.ScalarMappable(cmap=cmap,
                               norm=plt.Normalize(vmin=vmin,
                                                  vmax=vmax))
    
    cbar = fig.colorbar(im,cax=cax)
    
    cbar.outline.set_edgecolor('silver')
    
    cbar.dividers.set_color('red')
    
    cbar.ax.tick_params(labelsize=20) 
    
    cbar.minorticks_on()
    
    cax.yaxis.tick_left()
    
    cbar.set_label(label,font={'size':25})
    


def draw_attr_map(fig,ax,geo,col,cmap,vmax,vmin,
                  cbar_label,cbar_loc,size=None,
                  draw_edge=False,crs='epsg:4326'):
    if draw_edge:
        geo.to_crs(crs).plot(ax=ax,
                             column=col,
                             cmap=cmap,
                             vmax=vmax,
                             vmin=vmin,
                             edgecolor='white',
                             linewidth=0.5)
    else:
        geo.to_crs(crs).plot(ax=ax,
                column=col,
                cmap=cmap,
                vmax=vmax,
                vmin=vmin,
                markersize=size)
        
    
    add_colorbar(fig=fig,
                 ax=ax,
                 label=cbar_label,
                 vmax=vmax,
                 cmap=cmap,
                 loc=cbar_loc)
    
    

In [14]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25

crs = 'epsg:4326'

fig,ax = plt.subplots(figsize=(24,10),dpi=600)


draw_attr_map(fig=fig,
              ax=ax,
              geo=regn_attr,
              cmap='GnBu',
              col='Base',
              vmax=1.0,
              vmin=0.0,
              draw_edge=False,
              cbar_label='Degree of Electricity\n Self-sufficiency',
              cbar_loc=[0.05,0.275,0.02,0.4],
              crs=crs)

world.boundary.to_crs(crs).plot(ax=ax,
                                facecolor='none',
                                edgecolor='silver',
                                alpha=0.75)

ax.axis('off')

ax.set_xticks([],[])
ax.set_yticks([],[])

plt.savefig('./fig/ele_in_rate_map.jpg',bbox_inches='tight')
plt.savefig('./fig/ele_in_rate_map.pdf',bbox_inches='tight')

plt.close()